In [ ]:
import os
from openai import OpenAI

# model="openai/gpt-oss-20b", 
client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key=os.getenv("OPENAI_API_KEY")
)

In [ ]:
response = client.responses.create(
    model="llama3.2:latest",
    input="What to say Hello in Hindi language ?"
)

print(response.output_text)

#### Analyze images and files

[Link](https://developers.openai.com/api/docs/quickstart#analyze-images-and-files)

In [ ]:
#
response = client.responses.create(
    model="llama3.2:latest",
    input=[{
            "role": "user",
            "content": [{
                    "type": "input_text",
                    "text": "India and England are playing cricket match",
                },{
                    "type": "input_text",
                    "text": "Who are playing match ?",
                }]
        }]
)

print(response.output_text)

#### Extend the model with tools



##### Web Search

In [ ]:

response = client.responses.create(
    model="llama3.2:latest",
    tools=[{"type": "web_search"}],
    input="What was a positive news story from today?"
)

print(response.output_text)

##### File Search

In [ ]:
#
response = client.responses.create(
    model="llama3.2:latest",
    input="What is deep research by OpenAI?",
    tools=[{
        "type": "file_search",
        "vector_store_ids": ["<vector_store_id>"]
    }]
)
print(response)

##### Functional Call

In [ ]:
tools = [{
        "type": "function",
        "name": "get_weather",
        "description": "Get current temperature for a given location.",
        "parameters": {
            "type": "object",
            "properties": {
                "location": {
                    "type": "string",
                    "description": "City and country e.g. Bogotá, Colombia",
                }
            },
            "required": ["location"],
            "additionalProperties": False,
        },
        "strict": True,
    }]

response = client.responses.create(
    model="llama3.2:latest",
    input=[
        {"role": "user", "content": "What is the weather like in Paris today?"},
    ],
    tools=tools,
)

print(response.output[0].to_json())

##### Remote MCP Server


In [ ]:
#
resp = client.responses.create(
    model="llama3.2:latest",
    tools=[
        {
            "type": "mcp",
            "server_label": "dmcp",
            "server_description": "A Dungeons and Dragons MCP server to assist with dice rolling.",
            "server_url": "https://dmcp-server.deno.dev/sse",
            "require_approval": "never",
        },
    ],
    input="Roll 2d4+1",
)

print(resp.output_text)

##### Stream responses and build realtime apps



In [ ]:
#
stream = client.responses.create(
    model="llama3.2:latest",
    input=[
        {
            "role": "user",
            "content": "Say 'double bubble bath' ten times fast.",
        },
    ],
    stream=True,
)

for event in stream:
    print(event)

##### from openai import OpenAI


In [ ]:
# pip install -U openai-agents

from agents import Agent, Runner, Model, RunConfig
import asyncio


spanish_agent = Agent(
    name="Spanish agent",
    instructions="You only speak Spanish.",
    model="llama3.2:latest",
)

english_agent = Agent(
    name="English agent",
    instructions="You only speak English",
    model="llama3.2:latest",
)

triage_agent = Agent(
    name="Triage agent",
    instructions="Handoff to the appropriate agent based on the language of the request.",
    handoffs=[spanish_agent, english_agent],
    model="llama3.2:latest",
)


async def main():
    result = await Runner.run(triage_agent, input="Hola, ¿cómo estás?")
    print(result.final_output)


import nest_asyncio
nest_asyncio.apply()
asyncio.run(main()) # Now will work

#if __name__ == "__main__":
#    asyncio.run(main())

##### OpenAI Embeddings

In [ ]:
from openai import OpenAI

client = OpenAI(
    base_url='http://localhost:11434/v1/',
    #api_key='ollama'
)

response = client.embeddings.create(
    model="nomic-embed-text", # Ensure you have pulled this model: ollama pull nomic-embed-text
    input="The sky is blue because of Rayleigh scattering"
)

print(response.data[0].embedding)

[0.023892084136605263, 0.022229965776205063, -0.15269239246845245, -0.010197603143751621, 0.056493304669857025, 0.08489725738763809, -0.004279041662812233, 0.004624337423592806, 0.028653539717197418, -0.06537465006113052, 0.035416483879089355, 0.075495146214962, 0.07535520941019058, 0.07798031717538834, 0.017395099624991417, 0.026186540722846985, 0.0024474177043884993, -0.03594614565372467, 0.007194170728325844, 0.01597464643418789, -0.06760857254266739, -0.03763744980096817, 0.012154260650277138, -0.056366216391325, 0.053398966789245605, 0.07398132234811783, -0.015732785686850548, -0.013338346034288406, -0.0020769748371094465, -0.026457911357283592, 0.04968023672699928, -0.03772210702300072, -0.02052517607808113, -0.042442578822374344, -0.025650840252637863, -0.024173231795430183, 0.05399797111749649, -0.002309124916791916, 0.01461815182119608, -0.002340479986742139, 0.022236844524741173, -0.022602083161473274, -0.012697004713118076, -0.013832111842930317, 0.017284724861383438, -0.006

: 

In [ ]:
from langchain.agents import create_agent

def get_weather(city: str) -> str:
    """Get weather for a given city."""
    return f"It's always sunny in {city}!"

agent = create_agent(
    model="ollama:llama3.2:latest",
    tools=[get_weather],
    system_prompt="You are a helpful assistant",
)

# Run the agent
response = agent.invoke(
    {"messages": [{"role": "user", "content": "what is the weather in sf"}]}
)

for m in response['messages'] :
    m.pretty_print()
